## 2024数据探索
本ipython notebook使用3种方法(Gradient Boosting regression，XGBoost和线性回归)在2024数据上小试验一把。

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

### 1. 导入2024小数据集

In [ ]:
loads = pd.read_csv('load2024.csv')
weather = pd.read_csv('weather2024.csv')

#### 格式化时间列

In [ ]:
loads.timestamp = pd.to_datetime(loads.timestamp)
weather['timestamp'] = pd.to_datetime(weather.timeest)

In [ ]:
# 去掉气象要素缺失的记录，不然线性回归会报错
weather = weather.dropna(subset=['temperaturef', 'dewpointf', 'humidity', 'sealevelpressurein', 'windspeedkmh'])
# 去掉重复的时间
weather = weather.drop_duplicates(subset=['timestamp'])

### 2. 用最近邻对齐天气
天气信息的频度是小时级别的，我们载入的2012数据是每5分钟的间隔。下面这个函数实际上就是使用KNN去补全5分钟级别数据里的天气信息。

In [28]:
from sklearn.neighbors import NearestNeighbors

def find_nearest(group,match,groupname):
    """
    group: 需要匹配的组
    match: 用于匹配的组
    """
    # 1. 创建 NearestNeighbors 对象，设置参数 n_neighbors=1，表示只找最近的一个邻居,拟合 match 数据。.fit(...)：让查找器记住所有天气时间点。
    nbrs = NearestNeighbors(n_neighbors=1).fit(match['timestamp'].values[:,None]) # [:None]等价于.reshape(-1,1)
    
    # 2. 找到 group 中每个时间点在 match 中的最近邻居
    # nbrs.kneighbors(...)：对每个负荷时间点找最近的天气时间点，返回距离 dist 和下标 ind，形状都是 (n, 1)。
    distances, indices = nbrs.kneighbors(group['timestamp'].values[:,None]) # [:None]等价于.reshape(-1,1)
    
    # 3. 返回 match 中对应的最近邻居的索引
    # ind.ravel()：压平成一维；values[下标数组] 一次取出所有对应的天气时间。
    # group['nearesttime'] = ...：新增一列，记录每条负荷对应的最近天气时间。
    group['nearesttime'] = match['timestamp'].values[indices.ravel()]
    # group['nearest_index'] = indices.flatten() # flatten() 将二维数组展平为一维数组

    return group

In [ ]:
loads = find_nearest(loads, weather, 'timestamp')

In [ ]:
full = loads.merge(weather, left_on='nearesttime', right_on='timestamp')

#去除冗余列，重命名部分列 
full = full[['timestamp_x', 'load', 'nearesttime', 'temperaturef', \
            'dewpointf', 'humidity', 'sealevelpressurein', 'winddirdegrees', 'windspeedkmh', \
            'precipitationmm']].rename(columns={'timestamp_x': 'timestamp', 'nearesttime':'weathertime'})
full.columns

Index(['timestamp', 'load', 'weathertime', 'temperaturef', 'dewpointf',
       'humidity', 'sealevelpressurein', 'winddirdegrees', 'windspeedkmh',
       'precipitationmm'],
      dtype='str')

导出完整文件到csv文件中

In [32]:
full.to_csv('full2024.csv', index=False)

### 3. 构造特征
这是一个时间序列上的回归问题，需要在时间上做一些特征，可参照论文[Barta et al. 2015](http://arxiv.org/pdf/1506.06972.pdf)提到的方式，去构造细粒度的时间特征，上面那篇论文的应用场景也是用概率模型预测电价。构造的特征如下：<br>

    `dow`: day of the week (integer 0-6)
    `doy`: day of the year (integer 0-365)
    `day`: day of the month (integer 1-31)
    `woy`: week of the year (integer 1-52)
    `month`: month of the year (integer 1-12)
    `hour`: hour of the day (integer 0-23)
    `minute`: minute of the day (integer 0-1439)
    
    `t_m24`: load value from 24 hours earlier
    `t_m48`: load value from 48 hours earlier
    这里没有用到`tdif`: difference between load and t_m24，因为这会造成数据泄露，导致结果偏乐观。

In [ ]:
#取出n天前的电力需求
# 定义时间步长
pday = pd.Timedelta('1days')

def get_pre_daysload(x,n_days):
    try:
        lo = full[full.timestamp == x-n_days*pday].load.values[0]
    except:
        lo = full[full.timestamp == x].load.values[0]
    return lo
    

In [42]:
full['dow'] = full.timestamp.apply(lambda x: x.dayofweek)
full['doy'] = full.timestamp.apply(lambda x: x.dayofyear)
full['day'] = full.timestamp.apply(lambda x: x.day)
full['woy'] = full.timestamp.apply(lambda x: x.week)
full['month'] = full.timestamp.apply(lambda x: x.month)
full['hour'] = full.timestamp.apply(lambda x: x.hour)
full['minute'] = full.timestamp.apply(lambda x: x.hour*60 + x.minute)

full['t_m24'] = full.timestamp.apply(get_pre_daysload, args=(1,))
full['t_m48'] = full.timestamp.apply(get_pre_daysload, args=(2,))

In [43]:
full.to_csv('full2024_features.csv', index=False)

### 4. Gradient Boosting Regression

In [ ]:
# 准备训练数据

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split # 用于随机切分，后面按时间切分，故没用到。
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error,r2_score

In [6]:
full = pd.read_csv('full2024_features.csv')
full.columns

Index(['timestamp', 'load', 'weathertime', 'temperaturef', 'dewpointf',
       'humidity', 'sealevelpressurein', 'winddirdegrees', 'windspeedkmh',
       'precipitationmm', 'dow', 'doy', 'day', 'woy', 'month', 'hour',
       'minute', 't_m24', 't_m48'],
      dtype='str')

因为这个notebook只是在探索2024这一年的数据，所以首先不能随机切分，随机切分让所有分数都偏乐观（train_test_split 是随机打乱切分的。对时间序列来说，测试集里某一时刻的前后 5 分钟很可能都在训练集里，分数也会偏乐观。），更严格的做法是按时间切，用前面的月份训练、后面的月份测试。另外，因为只有一年的数据，所以严格按照前面的月份训练，后面的月份测试会导致模型没有见过后几个月的数据，doy、month、woy 的取值都超出了训练范围,导致测试结果不佳。
因此，我们这里只用了气温等外生变量代替"记住季节"。让模型根据温度、时刻、星期几等特征来判断负荷，而不是靠记住"几月份是什么水平"。只要测试期的温度范围在训练期出现过（比如训练里有过类似的冷天），模型就能泛化。然后将默认参数和滚动窗口交叉验证（rolling-origin / expanding window）对比。滚动窗口交叉验证不做一次性切分，而是多折：用1-2月训练、验证第3-4个月，再用前1-4个月训练、验证第5-6个月，依次往后。这样每个季节都被评估到，得到的是更稳健的误差估计。

普通 K 折交叉验证：把数据分成 K 份（常用 5 或 10），轮流拿一份当验证、其余当训练。它不管时间顺序，会出现用未来预测过去，所以不适合时间序列。TimeSeriesSplit：验证段永远在训练段之后。默认训练段越来越长，严格叫“扩展窗口”；加上 max_train_size 让训练段长度固定，才是严格意义的“滑动窗口”。两者通常都被归为滚动验证。

#### (1) 朴素baseline和模型baseline（默认参数）

In [ ]:
# 未放入winddirdegrees，因为这一列有缺失值：天气表里有 252 条为空（无风时没有风向），合并后大约是三千行。放进 X 的话，岭回归和 OLS 会因为 NaN 报错。
# 风向是环形的，359 度和 1 度几乎是同一个方向，但数值上差了 358。直接当数字用，线性模型会理解错。
X1 = full[[\
          'temperaturef',\
          'dewpointf', \
          'humidity', \
          'sealevelpressurein', \
         'windspeedkmh', \
          'precipitationmm',\
          'dow',\
          'hour',\
         'minute',\
          't_m24', \
          't_m48', \
         ]]
y1 = full['load']

In [18]:
full['timestamp'] = pd.to_datetime(full['timestamp'])
split_time = pd.Timestamp('2024-09-01')
train_mask = full.timestamp < split_time     # 9 月 1 日之前为 True

# train_mask 是一列 True/False，~ 可对pandas Series和NumPy 数组的bool值取反。前 8 个月训练、后 4 个月测试
X1_train, X1_test = X1[train_mask], X1[~train_mask]
y1_train, y1_test = y1[train_mask], y1[~train_mask]

朴素baseline

In [34]:
rows_t_m24 = []

# 改动 1：加一行朴素 baseline 作参照
rows_t_m24.append({
    '模型': '朴素(t_m24)',
    '训练集RMSE': np.sqrt(mean_squared_error(y1_train, X1_train['t_m24'])),
    '测试集RMSE': np.sqrt(mean_squared_error(y1_test, X1_test['t_m24'])),
    '测试集MAE': mean_absolute_error(y1_test, X1_test['t_m24']),
    '测试集R2': r2_score(y1_test, X1_test['t_m24']),
    '测试集MAPE(%)': mean_absolute_percentage_error(y1_test, X1_test['t_m24']) * 100,
})
rows_t_m24

[{'模型': '朴素(t_m24)',
  '训练集RMSE': np.float64(130.6438188226371),
  '测试集RMSE': np.float64(103.47844515345297),
  '测试集MAE': 72.61498492855745,
  '测试集R2': 0.7424064766964091,
  '测试集MAPE(%)': 5.77542056004978}]

In [20]:
# 梯度提升是依次训练很多棵小决策树，每棵树去拟合前面所有树的残差。默认 100 棵树、学习率 0.1、树深 3。
# loss='squared_error'：用平方误差做损失函数。
# verbose=1：训练时打印进度，就是输出里那张 Iter / Train Loss 表。
# warm_start=True：再次调用 fit 时在已有的树上继续加，而不是从头训练。
# .fit(X1, y1) 训练；.score(X1, y1) 返回 R²，越接近 1 越好。
gbr = GradientBoostingRegressor(loss='squared_error', verbose=1,warm_start=True)
gbr_fitted = gbr.fit(X1_train,y1_train)


      Iter       Train Loss   Remaining Time 
         1       60615.6608           11.50s
         2       52230.0288           10.87s
         3       45301.6702           11.09s
         4       39625.0224           10.78s
         5       34835.9206           10.62s
         6       30877.6219           10.50s
         7       27639.1007           10.56s
         8       24967.2008           10.53s
         9       22719.5106           10.53s
        10       20886.7048           10.42s
        20       11282.8644            9.48s
        30        8453.6278            8.31s
        40        7202.1135            7.15s
        50        6553.1040            6.02s
        60        6139.5149            4.93s
        70        5830.6458            3.77s
        80        5587.4487            2.55s
        90        5388.6902            1.28s
       100        5205.6915            0.00s


R² 衡量的是：模型比“永远猜平均值”好多少。R² = 1 − MSE ÷ y 的方差。
R²=1，预测完全准确；
0.9175，负荷的波动里，模型解释了约 92%；
0，和永远猜平均值一样；
负数，比猜平均值还差。R²没有单位，方便跨数据集比较；缺点是看不出具体差多少兆瓦。

In [24]:
# R² 模型解释了多少波动
print('R²:', gbr.score(X1_test,y1_test))

pred = gbr.predict(X1_test)
# 误差平方的均值再开方，单位是兆瓦，直观；对大误差惩罚重。和模型训练时的平方误差目标一致，适合做主指标。
print('RMSE:', np.sqrt(mean_squared_error(y1_test, pred)))

# 误差绝对值的均值，单位是兆瓦，表示平均差多少；不容易被个别大误差带偏。
print('MAE :', mean_absolute_error(y1_test, pred))

# MAPE 是“误差占真实值的百分比”的平均，比如 3% 表示平均偏差是真实负荷的 3%。
print('MAPE:', mean_absolute_percentage_error(y1_test, pred) * 100, '%')

R²: 0.8373683611666379
RMSE: 82.22141451751321
MAE : 61.37464759524987
MAPE: 4.760973258791568 %


In [25]:
# R² 模型解释了多少波动
print('R²:', gbr.score(X1_train,y1_train))

pred = gbr.predict(X1_train)
print('RMSE:', np.sqrt(mean_squared_error(y1_train, pred)))
print('MAE :', mean_absolute_error(y1_train, pred))
print('MAPE:', mean_absolute_percentage_error(y1_train, pred) * 100, '%')

R²: 0.9265945659351049
RMSE: 72.15047835652274
MAE : 54.12339021404018
MAPE: 4.342464538069831 %


#### (2) 加入扩展窗口交叉验证，选最优参数

In [30]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid={
        'max_depth': [2, 3],            # 树浅一些
        'min_samples_leaf': [50, 200],  # 每个叶子至少这么多样本，避免记住个别时刻
        'subsample': [0.7, 1.0],        # 每棵树只用一部分行
        'learning_rate': [0.05, 0.1],   # 每棵树的贡献小一些
    },
    cv=TimeSeriesSplit(n_splits=3),
    scoring='neg_root_mean_squared_error',
)
search.fit(X1_train, y1_train)      # 只用训练集

print(search.best_params_)               # 选出的最优参数
best_model = search.best_estimator_   # 已用最优参数在整个训练集上重新训练好

{'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 200, 'subsample': 0.7}


In [31]:
print('R²:', best_model.score(X1_test,y1_test))

pred = best_model.predict(X1_test)
print('RMSE:', np.sqrt(mean_squared_error(y1_test, pred)))
print('MAE :', mean_absolute_error(y1_test, pred))
print('MAPE:', mean_absolute_percentage_error(y1_test, pred) * 100, '%')

R²: 0.8404284826317481
RMSE: 81.4441907121431
MAE : 60.851767652580975
MAPE: 4.714390463534486 %


In [32]:
print('R²:', best_model.score(X1_train,y1_train))

pred = best_model.predict(X1_train)
print('RMSE:', np.sqrt(mean_squared_error(y1_train, pred)))
print('MAE :', mean_absolute_error(y1_train, pred))
print('MAPE:', mean_absolute_percentage_error(y1_train, pred) * 100, '%')

R²: 0.9272722431525605
RMSE: 71.81666037201853
MAE : 53.83578229469646
MAPE: 4.305381780700534 %


#### (3) 结论

使用3折交叉验证找到的最优参数和默认参数在树深和学习率上一样，只有'min_samples_leaf': 200, 'subsample': 0.7和默认的1，1不同，但是提升很小。

### 5. 与其他模型的对比
这里XGBoost、不带L2正则的LinearRegression和带正则的LinearRegression用同一份训练集训练、同一份测试集评估、同一个指标，和GBR对比。

#### (1) 模型baseline（默认参数）

In [35]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# XGBoost 的三个参数默认和 GBR 的默认值一样（100 棵树、学习率 0.1、树深 3）。
# StandardScaler：L2 惩罚的是系数大小，而系数大小取决于特征的量纲（气压是 30 左右，minute 能到 1439）。先标准化，惩罚对各特征才公平。
# make_pipeline 把“标准化 + 岭回归”串成一个模型，并保证标准化的均值和方差只从训练集计算。
models = {
    'GBR': GradientBoostingRegressor(loss='squared_error', random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    '线性回归(无正则)': LinearRegression(),
    '线性回归(L2正则)': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
}

rows = []
for name, model in models.items():
    model.fit(X1_train, y1_train)            # 都只在训练集上训练
    pred_train = model.predict(X1_train)
    pred_test = model.predict(X1_test)      # 都在同一个测试集上预测
    rows.append({
        '模型': name,
        '训练集RMSE': np.sqrt(mean_squared_error(y1_train, pred_train)),
        '测试集RMSE': np.sqrt(mean_squared_error(y1_test, pred_test)),
        '测试集MAE': mean_absolute_error(y1_test, pred_test),
        '测试集R2': r2_score(y1_test, pred_test),
        '测试集MAPE(%)': mean_absolute_percentage_error(y1_test, pred_test) * 100,
    })

pd.DataFrame(rows).sort_values('测试集RMSE')

,模型,训练集RMSE,测试集RMSE,测试集MAE,测试集R2,测试集MAPE(%)
1,XGBoost,71.537636,81.656700,61.103786,0.839595,4.743667
0,GBR,72.150478,82.221415,61.374648,0.837368,4.760973
3,线性回归(L2正则),120.425924,103.675501,76.861636,0.741424,6.086207
2,线性回归(无正则),120.425910,103.679850,76.863297,0.741403,6.086314


#### (2) 加入扩展窗口交叉验证，选最优参数

In [44]:
cv = TimeSeriesSplit(n_splits=3)
rmse = 'neg_root_mean_squared_error'

models = {
    'XGBoost(调参)': GridSearchCV(XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
                                 {'max_depth': [2, 3], 'min_child_weight': [1, 50]}, cv=cv, scoring=rmse),
    '线性回归(无正则)': LinearRegression(),      # 没有可调的参数
    '线性回归(L2正则)': GridSearchCV(make_pipeline(StandardScaler(), Ridge()),
                                    {'ridge__alpha': [0.1, 1, 10, 100]}, cv=cv, scoring=rmse),
}

rows_adj = []
for name, model in models.items():
    model.fit(X1_train, y1_train)            # 都只在训练集上训练
    if hasattr(model, 'best_params_'):       # 只有 GridSearchCV 才有这个属性
        print(name, '最优参数:', model.best_params_, '验证集平均RMSE:', -model.best_score_)
    pred_train = model.predict(X1_train)
    pred_test = model.predict(X1_test)      # 都在同一个测试集上预测
    rows_adj.append({
        '模型': name,
        '调参后的训练集RMSE': np.sqrt(mean_squared_error(y1_train, pred_train)),
        '调参后的测试集RMSE': np.sqrt(mean_squared_error(y1_test, pred_test)),
        '调参后的测试集MAE': mean_absolute_error(y1_test, pred_test),
        '调参后的测试集R2': r2_score(y1_test, pred_test),
        '调参后的测试集MAPE(%)': mean_absolute_percentage_error(y1_test, pred_test) * 100,
    })

# 第四节调好的 GBR：先明确算出它自己的预测
print('GBR(调参) 最优参数:', search.best_params_, '验证集平均RMSE:', -search.best_score_)
gbr_train_adj = best_model.predict(X1_train)
gbr_test_adj = best_model.predict(X1_test)
rows_adj.append({'模型': 'GBR(调参)',
        '调参后的训练集RMSE': np.sqrt(mean_squared_error(y1_train, gbr_train_adj)),
        '调参后的测试集RMSE': np.sqrt(mean_squared_error(y1_test, gbr_test_adj)),
        '调参后的测试集MAE': mean_absolute_error(y1_test, gbr_test_adj),
        '调参后的测试集R2': r2_score(y1_test, gbr_test_adj),
        '调参后的测试集MAPE(%)': mean_absolute_percentage_error(y1_test, gbr_test_adj) * 100})

pd.DataFrame(rows_adj).sort_values('调参后的测试集RMSE')

XGBoost(调参) 最优参数: {'max_depth': 3, 'min_child_weight': 1} 验证集平均RMSE: 134.90506834184575
线性回归(L2正则) 最优参数: {'ridge__alpha': 10} 验证集平均RMSE: 155.99801651534378
GBR(调参) 最优参数: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 200, 'subsample': 0.7} 验证集平均RMSE: 131.10682783930906


,模型,调参后的训练集RMSE,调参后的测试集RMSE,调参后的测试集MAE,调参后的测试集R2,调参后的测试集MAPE(%)
3,GBR(调参),71.816660,81.444191,60.851768,0.840428,4.714390
0,XGBoost(调参),71.537636,81.656700,61.103786,0.839595,4.743667
2,线性回归(L2正则),120.427139,103.639486,76.848869,0.741604,6.085408
1,线性回归(无正则),120.425910,103.679850,76.863297,0.741403,6.086314


#### (3) 结论

GBR和XGBoost在该时序预测上RMSE明显优于线性回归，没有出现明显过拟合，但调参后无明显提升。

### 6. 总结

#### (1) 这个 notebook 做了什么

用 2024 年一年的数据，试验“用天气和历史负荷预测当前负荷”这件事能做到什么程度。

1. **数据**：`load2024.csv`（5 分钟一条的负荷）和 `weather2024.csv`（约每小时一条的天气），时间都是纽约当地时间。
2. **对齐**：用最近邻（`NearestNeighbors`）给每条负荷找时间上最近的一条天气记录，合并成一张表，共 107338 行。
3. **特征**：6 个天气量（气温、露点、湿度、气压、风速、降水）、3 个时间量（`dow`、`hour`、`minute`）、2 个历史负荷（`t_m24`、`t_m48`），共 11 个。
4. **切分**：按时间切，1–8 月训练（71575 行），9–12 月测试（35763 行）。
5. **模型**：朴素 baseline（直接拿 `t_m24` 当预测）、GBR、XGBoost、线性回归、岭回归；树模型和岭回归各做了一次默认参数、一次用 `TimeSeriesSplit` 3 折交叉验证调参。

和原 notebook 相比，有三处有意的不同：

- **去掉了 `tdif`**：它是用要预测的 `load` 算出来的，属于数据泄露。
- **不用随机切分**：相邻 5 分钟的数据几乎相同，随机切分会让分数偏乐观（随机切分时 GBR 的 R² 是 0.9175，按时间切后是 0.837）。
- **不用 `doy`、`woy`、`month`**：只有一年数据时，测试集的月份在训练集里没出现过。

#### (2) 结果

| 模型 | 训练集 RMSE | 测试集 RMSE | 测试集 MAE | 测试集 R² | 测试集 MAPE |
|---|---|---|---|---|---|
| GBR（调参） | 71.8 | **81.4** | 60.9 | 0.840 | 4.71% |
| XGBoost（默认，调参后参数不变） | 71.5 | 81.7 | 61.1 | 0.840 | 4.74% |
| GBR（默认） | 72.2 | 82.2 | 61.4 | 0.837 | 4.76% |
| 朴素（`t_m24`） | 130.6 | 103.5 | 72.6 | 0.742 | 5.78% |
| 岭回归（调参，alpha=10） | 120.4 | 103.6 | 76.8 | 0.742 | 6.09% |
| 线性回归 | 120.4 | 103.7 | 76.9 | 0.741 | 6.09% |

RMSE 和 MAE 的单位是兆瓦。

#### (3) 结论

- **树模型有效。** GBR 和 XGBoost 的测试 RMSE 比朴素 baseline 低约 21%，MAPE 从 5.8% 降到 4.7%。
- **GBR 和 XGBoost 没有差别。** 两者测试 RMSE 只差 0.2–0.8，远小于各折之间的波动，不能据此说哪个更好。
- **线性模型没有用。** 线性回归和岭回归的测试 RMSE 与朴素 baseline 几乎一样（MAE 还略差），等于没有从天气里学到额外的东西。可能的原因是气温和负荷不是直线关系（太冷太热负荷都高），线性模型表达不了。
- **L2 正则没有作用。** 只有 11 个特征、7 万行数据，线性回归本来就不会过拟合，加不加正则结果相同。
- **调参几乎没有收益。** GBR 调参后测试 RMSE 只从 82.2 降到 81.4，XGBoost 选出的就是默认参数。瓶颈不在参数上。
- **过拟合不明显。** 树模型训练 RMSE 约 72、测试约 82，差距不大，其中一部分来自训练期（含夏季）和测试期（秋冬）本身的差别。

#### (4) 读这些数字时要注意的地方

- **训练集和测试集的难度不同，训练误差和测试误差不能直接比。** 训练期负荷的标准差是 266，测试期是 204；朴素 baseline 在训练集上的 RMSE（130.6）也高于测试集（103.5）。线性回归训练误差高于测试误差就是这个原因，不是出错。
- **交叉验证的分数不稳定。** 验证集平均 RMSE 是 131–156，远高于测试集的 81–104，因为 3 折分别是用冬天预测春天、用冬春预测初夏、用上半年预测盛夏，每一折都在预测没见过的季节。一年数据上的交叉验证只能粗略参考。
- **`t_m24` 有少量泄露。** 约 2% 的行（2286 行）找不到 24 小时前的记录，`get_pre_daysload` 退回去取了当前时刻的负荷。影响不大，但严格来说这些行应删掉。
- **测试集只代表 2024 年 9–12 月。** 换一个时间段，数字会不同。

#### (5) 下一步

- 换成 2021–2024 多年数据训练、2025 年测试。那时每个月份都见过几次，可以把 `doy`、`month` 等季节特征加回来（另行试验过：一年数据时加入它们，GBR 的测试 RMSE 会从 82 升到 117）。
- 处理 `t_m24` 的回退问题，找不到历史记录的行直接丢弃。
- 最终和 NYISO 官方预测比较，看差距有多大。